# Análise Exploratória — KOI Cumulative

Este notebook explora a Camada Gold do NASA Exoplanet Archive. Execute a ingestão e `python -m src.gold` antes de iniciar.

Objetivo: entender a distribuição de `koi_disposition`, qualidade dos dados e relações entre variáveis físicas.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from src.config import DATABASE_URL, GOLD_SCHEMA

engine = create_engine(DATABASE_URL, pool_pre_ping=True)
with engine.connect() as conn:
    df = pd.read_sql_query(text(f'SELECT * FROM {GOLD_SCHEMA}.koi_analytics'), conn)
df.head()

## Visão geral e qualidade

In [ ]:
print('Dimensões:', df.shape)
display(df.describe(include='all').T)
missing = df.isna().mean().sort_values(ascending=False).mul(100)
display(missing.to_frame('missing_pct').head(20))

## Distribuição da variável alvo

In [ ]:
plt.figure(figsize=(9, 5))
order = df['koi_disposition'].value_counts().index
sns.countplot(data=df, x='koi_disposition', order=order)
plt.title('Distribuição de koi_disposition')
plt.xlabel('Classificação')
plt.ylabel('Quantidade')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
display(df['koi_disposition'].value_counts(normalize=True).mul(100).round(2))

## Distribuição das variáveis físicas

In [ ]:
numeric_cols = [c for c in ['koi_period','koi_prad','koi_teq','koi_depth','koi_model_snr','koi_steff'] if c in df.columns]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    plt.figure(figsize=(9, 4))
    sns.histplot(data=df, x=col, hue='koi_disposition', element='step', stat='density', common_norm=False)
    plt.title(f'Distribuição de {col} por classe')
    plt.tight_layout()
    plt.show()

## Correlação entre atributos numéricos

In [ ]:
corr = df.select_dtypes(include=np.number).corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Matriz de correlação')
plt.tight_layout()
plt.show()

## Considerações para o treinamento

- A variável alvo é `koi_disposition` (CONFIRMED, CANDIDATE e FALSE POSITIVE).
- A distribuição das classes deve ser considerada no split e na avaliação.
- Variáveis de disposição derivada e flags de falso positivo são excluídas para reduzir vazamento de alvo.
- Imputação e escalonamento são ajustados dentro de pipelines, somente com os dados de treino.